# Example 11 — Gaussian external constraints

This notebook shows how an external measurement can constrain a fit parameter while preserving the Dalitz likelihood.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from dalitzplotfitter import (BackgroundCategory, ConstrainedNLL, DecayChannel, DecayModel, GaussianConstraint, Minimizer, MultiBackgroundNLL, NonResonant, Parameter, PhaseSpaceSample, RealImag, Resonance, enable_x64, weighted_resample)
from dalitzplotfitter.background import FunctionalBackground
enable_x64()
channel=DecayChannel('B+',('K+','pi+','pi-'))
model=DecayModel(channel,[Resonance('Kstar892',(0,2),RealImag(1,0),mass=0.8958,width=0.0474,spin=1),Resonance('rho770',(1,2),RealImag(0.65,0.10),mass=0.7753,width=0.1491,spin=1),NonResonant(RealImag(-0.5,0.1))],normalization_method='square-dalitz',normalization_resolution=250,normalization_pair=(0,2))


## 1. Generate a signal + background toy


In [ ]:
pool=model.generate_phase_space(100000,seed=11001); norm=model.normalization_sample
bkg=FunctionalBackground(lambda d:0.6+0.5*(d['s23']-jnp.min(norm.s23))/(jnp.max(norm.s23)-jnp.min(norm.s23)))
N=12000; FS_TRUE=0.68; ns=int(round(N*FS_TRUE)); nb=N-ns
sig=weighted_resample(jax.random.key(11002),pool,pool.weights*model.intensity(pool.as_dict()),ns,replace=True)
bg=weighted_resample(jax.random.key(11003),pool,pool.weights*bkg(pool.as_dict()),nb,replace=True)
def merge(a,b):
    def c(name):
        x,y=getattr(a,name),getattr(b,name); return None if x is None else jnp.concatenate((x,y))
    return PhaseSpaceSample(s12=c('s12'),s13=c('s13'),s23=c('s23'),weights=jnp.ones(a.size+b.size),p1=c('p1'),p2=c('p2'),p3=c('p3'))
data=merge(sig,bg); d=data.as_dict()
signal_pdf=model.pdf(); bnorm=jnp.mean(norm.weights*bkg(norm.as_dict()))
f_sig=Parameter('signal_fraction',0.55,bounds=(0.01,0.99),step=0.01)
cat=BackgroundCategory('comb',bkg(d),bnorm)
base_nll=MultiBackgroundNLL(signal_density=lambda v:signal_pdf(d,v),backgrounds=(cat,),signal_fraction=f_sig)


## 2. Add an external constraint

Suppose an independent mass fit measures $f_{sig}=0.70\pm0.04$. The penalty is $\frac12[(f_{sig}-0.70)/0.04]^2$.


In [ ]:
constraint=GaussianConstraint(f_sig,mean=0.70,sigma=0.04)
constrained_nll=ConstrainedNLL(base_nll,constraint)
start={'signal_fraction':0.55}
fit_free=Minimizer(base_nll,(f_sig,),verbose=0).fit(start_values=start,simplex=True,ncall=10000)
fit_con=Minimizer(constrained_nll,(f_sig,),verbose=0).fit(start_values=start,simplex=True,ncall=10000)
free=float(fit_free.values['signal_fraction']); con=float(fit_con.values['signal_fraction'])
print('truth:',FS_TRUE)
print('start:',start['signal_fraction'])
print('unconstrained fit:',free)
print('constrained fit:',con)


## 3. NLL scan: data likelihood versus constrained likelihood


In [ ]:
grid=np.linspace(0.52,0.82,180)
n0=np.array([float(base_nll({'signal_fraction':x})) for x in grid]); n1=np.array([float(constrained_nll({'signal_fraction':x})) for x in grid])
n0-=n0.min(); n1-=n1.min()
plt.figure(figsize=(7,5)); plt.plot(grid,n0,label='Dalitz likelihood'); plt.plot(grid,n1,label='Dalitz + Gaussian constraint'); plt.axvline(FS_TRUE,ls='--',label='generated truth'); plt.axvline(0.70,ls=':',label='external central value'); plt.xlabel(r'$f_{sig}$'); plt.ylabel(r'$\Delta$NLL'); plt.ylim(0,12); plt.legend(); plt.show()


## 4. Dalitz plot used by the fit


In [ ]:
plt.figure(figsize=(7,5.5)); h=plt.hist2d(np.asarray(data.s13),np.asarray(data.s23),bins=65); plt.colorbar(h[3]); plt.xlabel(r'$s_{13}$ [GeV$^2$]'); plt.ylabel(r'$s_{23}$ [GeV$^2$]'); plt.title('Toy data for the constrained fit'); plt.show()
